# S4-01: RAG 기초 — 청킹, 임베딩, ChromaDB
**Skilljar L01-L03: Introducing RAG / Text Chunking / Text Embeddings**

## 학습 목표
- RAG의 개념과 필요성을 이해한다
- 3가지 청킹 전략 (고정 크기, 의미 기반, 재귀적)을 구현한다
- 텍스트 임베딩과 코사인 유사도를 직접 계산한다
- ChromaDB에 문서를 저장하고 유사도 검색을 수행한다

## 사전 준비
1. `.env` 파일에 API 키 설정:
```
ANTHROPIC_API_KEY="YOUR_API_KEY_HERE"
OPENAI_API_KEY="sk-your-openai-key-here"
```

In [ ]:
# 패키지 설치
%pip install anthropic openai python-dotenv chromadb numpy

In [ ]:
# 환경변수 로드
from dotenv import load_dotenv
load_dotenv()

In [ ]:
# 클라이언트 생성
from anthropic import Anthropic
from openai import OpenAI
import chromadb
import numpy as np

anthropic_client = Anthropic()
openai_client = OpenAI()
chroma_client = chromadb.Client()
model = "claude-sonnet-4-0"

print("모든 클라이언트 초기화 완료")

---
## Exercise 1: 텍스트 청킹 전략 구현

3가지 청킹 전략을 구현하고 동일한 텍스트에 적용하여 결과를 비교하세요.

**요구사항:**
1. `fixed_size_chunk(text, chunk_size, overlap)` 함수 구현
2. `semantic_chunk(text)` 함수 구현 (빈 줄 기준)
3. `recursive_chunk(text, max_size, separators)` 함수 구현
4. 각 함수의 결과를 출력하고 비교

**테스트 텍스트:**
```python
sample = """4.2.1 일반 사항
콘크리트구조 부재의 설계는 극한강도설계법에 따른다. 모든 부재는 소요강도 이상의 설계강도를 가져야 한다.

4.2.2 강도감소계수
인장지배 단면의 강도감소계수는 0.85로 한다. 압축지배 단면의 경우 나선철근 부재는 0.70, 기타 부재는 0.65로 한다.

4.3.1 축하중을 받는 부재
축방향 압축력을 받는 부재의 공칭강도 Pn = 0.80 * [0.85 * fck * (Ag - Ast) + fy * Ast]. 여기서 Ag는 전체 단면적이다.

4.4.1 최소 철근비
기둥의 종방향 철근비는 전체 단면적의 1% 이상, 8% 이하로 한다. 최소 4개의 종방향 철근을 배치해야 한다.
"""
```

**기대 출력 (예시):**
```
=== 고정 크기 청킹 (chunk_size=100, overlap=20) ===
청크 수: 5
[0] (100자) 4.2.1 일반 사항\n콘크리트구조 부재의 설계는...
...

=== 의미 기반 청킹 ===
청크 수: 4
[0] (66자) 4.2.1 일반 사항\n콘크리트구조 부재의...
...

=== 재귀적 청킹 (max_size=120) ===
청크 수: 5
[0] (66자) 4.2.1 일반 사항\n콘크리트구조 부재의...
...
```

In [ ]:
# 테스트 텍스트
sample = """4.2.1 일반 사항
콘크리트구조 부재의 설계는 극한강도설계법에 따른다. 모든 부재는 소요강도 이상의 설계강도를 가져야 한다.

4.2.2 강도감소계수
인장지배 단면의 강도감소계수는 0.85로 한다. 압축지배 단면의 경우 나선철근 부재는 0.70, 기타 부재는 0.65로 한다.

4.3.1 축하중을 받는 부재
축방향 압축력을 받는 부재의 공칭강도 Pn = 0.80 * [0.85 * fck * (Ag - Ast) + fy * Ast]. 여기서 Ag는 전체 단면적이다.

4.4.1 최소 철근비
기둥의 종방향 철근비는 전체 단면적의 1% 이상, 8% 이하로 한다. 최소 4개의 종방향 철근을 배치해야 한다.
"""

# TODO: 3가지 청킹 함수를 구현하세요

# 1. 고정 크기 청킹
# def fixed_size_chunk(text, chunk_size=100, overlap=20):

# 2. 의미 기반 청킹
# def semantic_chunk(text):

# 3. 재귀적 청킹
# def recursive_chunk(text, max_size=120, separators=None):

In [ ]:
# ===== 정답 =====

def fixed_size_chunk(text: str, chunk_size: int = 100, overlap: int = 20) -> list[str]:
    """고정 크기 청킹: 일정한 문자 수로 텍스트를 분할한다."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
    return chunks

def semantic_chunk(text: str) -> list[str]:
    """의미 기반 청킹: 빈 줄(문단)을 기준으로 분할한다."""
    paragraphs = text.split("\n\n")
    chunks = [p.strip() for p in paragraphs if p.strip()]
    return chunks

def recursive_chunk(text: str, max_size: int = 120, separators: list[str] = None) -> list[str]:
    """재귀적 청킹: 구분자 우선순위에 따라 계층적으로 분할한다."""
    if separators is None:
        separators = ["\n\n", "\n", ". ", " "]

    if len(text) <= max_size:
        return [text]

    for sep in separators:
        if sep in text:
            parts = text.split(sep)
            chunks = []
            current = ""
            for part in parts:
                candidate = current + sep + part if current else part
                if len(candidate) <= max_size:
                    current = candidate
                else:
                    if current:
                        chunks.append(current)
                    if len(part) > max_size:
                        sub_seps = separators[separators.index(sep) + 1:] if separators.index(sep) + 1 < len(separators) else [" "]
                        chunks.extend(recursive_chunk(part, max_size, sub_seps))
                        current = ""
                    else:
                        current = part
            if current:
                chunks.append(current)
            return chunks

    return fixed_size_chunk(text, max_size, overlap=0)

# 테스트 실행
print("=== 고정 크기 청킹 (chunk_size=100, overlap=20) ===")
fixed_chunks = fixed_size_chunk(sample, chunk_size=100, overlap=20)
print(f"청크 수: {len(fixed_chunks)}")
for i, c in enumerate(fixed_chunks):
    print(f"[{i}] ({len(c)}자) {c[:50].replace(chr(10), '\\n')}...")
print()

print("=== 의미 기반 청킹 ===")
sem_chunks = semantic_chunk(sample)
print(f"청크 수: {len(sem_chunks)}")
for i, c in enumerate(sem_chunks):
    print(f"[{i}] ({len(c)}자) {c[:50].replace(chr(10), '\\n')}...")
print()

print("=== 재귀적 청킹 (max_size=120) ===")
rec_chunks = recursive_chunk(sample, max_size=120)
print(f"청크 수: {len(rec_chunks)}")
for i, c in enumerate(rec_chunks):
    print(f"[{i}] ({len(c)}자) {c[:50].replace(chr(10), '\\n')}...")

def verify_ex1():
    assert len(fixed_size_chunk("a" * 250, chunk_size=100, overlap=20)) >= 3, "고정 크기 청킹 수가 부족합니다"
    assert len(semantic_chunk("A\n\nB\n\nC")) == 3, "의미 기반 청킹이 3개여야 합니다"
    assert all(len(c) <= 120 for c in recursive_chunk(sample, max_size=120)), "재귀적 청크가 max_size를 초과합니다"
    print("Exercise 1 통과!")

verify_ex1()

---
## Exercise 2: 코사인 유사도 계산

코사인 유사도 함수를 직접 구현하고, 건축공학 텍스트의 의미적 유사도를 측정하세요.

**요구사항:**
1. `cosine_similarity(vec_a, vec_b)` 함수를 numpy로 구현
2. 3개의 3차원 벡터로 기본 테스트 수행
3. OpenAI 임베딩 API를 사용하여 실제 텍스트의 유사도 측정

**테스트 텍스트:**
```python
texts = [
    "RC 기둥의 축하중 설계",          # 텍스트 A
    "콘크리트 기둥의 압축 강도 검토",     # 텍스트 B (A와 유사)
    "강구조 보의 횡좌굴 검토",          # 텍스트 C (A와 다름)
    "기둥 종방향 철근비 산정",          # 텍스트 D (A와 관련)
]
```

**기대 출력:**
```
=== 기본 벡터 테스트 ===
v1 vs v2 (유사): 0.9xxx
v1 vs v3 (다름): 0.3xxx

=== 실제 텍스트 임베딩 유사도 ===
A vs B: 0.8x (높음 — 둘 다 기둥 설계)
A vs C: 0.5x (낮음 — 기둥 vs 보)
A vs D: 0.7x (중간 — 기둥 관련이지만 다른 주제)
```

In [ ]:
# TODO: 코사인 유사도 함수를 구현하고 테스트하세요

texts = [
    "RC 기둥의 축하중 설계",
    "콘크리트 기둥의 압축 강도 검토",
    "강구조 보의 횡좌굴 검토",
    "기둥 종방향 철근비 산정",
]

# 1. cosine_similarity(vec_a, vec_b) 구현

# 2. 기본 벡터 테스트

# 3. 실제 임베딩 유사도 측정

In [ ]:
# ===== 정답 =====

def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    """두 벡터의 코사인 유사도를 계산한다."""
    a = np.array(vec_a)
    b = np.array(vec_b)
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return float(dot_product / (norm_a * norm_b))

# 기본 벡터 테스트
print("=== 기본 벡터 테스트 ===")
v1 = [1.0, 0.8, 0.2]
v2 = [0.9, 0.85, 0.15]
v3 = [0.1, 0.2, 0.95]

sim_12 = cosine_similarity(v1, v2)
sim_13 = cosine_similarity(v1, v3)
print(f"v1 vs v2 (유사): {sim_12:.4f}")
print(f"v1 vs v3 (다름): {sim_13:.4f}")

# 실제 텍스트 임베딩 유사도
print("\n=== 실제 텍스트 임베딩 유사도 ===")
texts = [
    "RC 기둥의 축하중 설계",
    "콘크리트 기둥의 압축 강도 검토",
    "강구조 보의 횡좌굴 검토",
    "기둥 종방향 철근비 산정",
]

response = openai_client.embeddings.create(
    input=texts,
    model="text-embedding-3-small"
)
embeddings = [item.embedding for item in response.data]

labels = ["A", "B", "C", "D"]
for i in range(1, len(texts)):
    sim = cosine_similarity(embeddings[0], embeddings[i])
    print(f"A vs {labels[i]}: {sim:.4f} — '{texts[0]}' vs '{texts[i]}'")

def verify_ex2():
    assert abs(cosine_similarity([1, 0], [1, 0]) - 1.0) < 0.001, "동일 벡터는 유사도 1.0"
    assert abs(cosine_similarity([1, 0], [0, 1]) - 0.0) < 0.001, "직교 벡터는 유사도 0.0"
    assert sim_12 > sim_13, "유사한 벡터의 유사도가 더 높아야 합니다"
    print("Exercise 2 통과!")

verify_ex2()

---
## Exercise 3: ChromaDB 벡터 데이터베이스 구축

ChromaDB에 KDS 설계기준 문서를 저장하고 유사도 검색을 수행하세요.

**요구사항:**
1. `kds_structural` 이름의 컬렉션을 생성
2. 제공된 8개 KDS 문서를 임베딩과 함께 저장
3. 3가지 질의에 대해 Top-3 검색을 수행
4. 검색 결과에서 문서 ID, 거리, 메타데이터를 출력

**테스트 질의:**
```python
queries = [
    "기둥의 최소 철근비는?",
    "보의 전단강도 계산식은?",
    "강도감소계수 0.85 적용 조건은?",
]
```

**기대 출력 (각 질의에 대해):**
```
질의: 기둥의 최소 철근비는?
[1] kds-4.4.1 (거리: 0.xxxx) 조항: 4.4.1 | 주제: 철근비
    4.4.1 최소 철근비: 기둥의 종방향 철근비는...
[2] ...
[3] ...
```

In [ ]:
# KDS 문서 데이터
documents = [
    {"id": "kds-4.2.1", "text": "4.2.1 일반 사항: 콘크리트구조 부재의 설계는 극한강도설계법에 따른다. 모든 부재는 소요강도 이상의 설계강도를 가져야 한다. 설계강도 = 강도감소계수(phi) x 공칭강도(Rn).", "metadata": {"clause": "4.2.1", "chapter": "4", "topic": "일반"}},
    {"id": "kds-4.2.2", "text": "4.2.2 강도감소계수: 인장지배 단면의 강도감소계수는 0.85로 한다. 압축지배 단면의 경우 나선철근 부재는 0.70, 기타 부재는 0.65로 한다.", "metadata": {"clause": "4.2.2", "chapter": "4", "topic": "강도감소계수"}},
    {"id": "kds-4.3.1", "text": "4.3.1 축하중을 받는 부재: 축방향 압축력을 받는 부재의 공칭강도 Pn = 0.80 * [0.85 * fck * (Ag - Ast) + fy * Ast]. 여기서 Ag는 전체 단면적, Ast는 철근 단면적이다.", "metadata": {"clause": "4.3.1", "chapter": "4", "topic": "축하중"}},
    {"id": "kds-4.4.1", "text": "4.4.1 최소 철근비: 기둥의 종방향 철근비는 전체 단면적의 1% 이상, 8% 이하로 한다. 최소 4개의 종방향 철근을 배치해야 한다.", "metadata": {"clause": "4.4.1", "chapter": "4", "topic": "철근비"}},
    {"id": "kds-4.4.2", "text": "4.4.2 띠철근 간격: 띠철근 간격은 다음 중 작은 값 이하로 한다. (1) 종방향 철근 지름의 16배 (2) 띠철근 지름의 48배 (3) 기둥 단면의 최소 치수.", "metadata": {"clause": "4.4.2", "chapter": "4", "topic": "띠철근"}},
    {"id": "kds-5.1.1", "text": "5.1.1 보의 휨 설계: 보의 공칭 휨강도 Mn = As * fy * (d - a/2). 여기서 a = As * fy / (0.85 * fck * b). 보의 인장 철근비는 균형 철근비의 75% 이하로 제한한다.", "metadata": {"clause": "5.1.1", "chapter": "5", "topic": "보 휨설계"}},
    {"id": "kds-5.2.1", "text": "5.2.1 보의 전단 설계: 콘크리트가 부담하는 전단강도 Vc = (1/6) * sqrt(fck) * b * d. 전단철근이 부담하는 전단강도 Vs = Av * fy * d / s.", "metadata": {"clause": "5.2.1", "chapter": "5", "topic": "보 전단설계"}},
    {"id": "kds-6.1.1", "text": "6.1.1 내진설계 일반: 내진설계범주 D 이상의 구조물에서 특수 모멘트골조를 사용하는 경우, 기둥의 강도는 보 강도의 1.2배 이상이어야 한다 (강기둥-약보 원칙).", "metadata": {"clause": "6.1.1", "chapter": "6", "topic": "내진설계"}},
]

queries = [
    "기둥의 최소 철근비는?",
    "보의 전단강도 계산식은?",
    "강도감소계수 0.85 적용 조건은?",
]

# TODO: ChromaDB 컬렉션을 생성하고, 문서를 저장하고, 검색하세요

In [ ]:
# ===== 정답 =====

def get_embeddings(texts: list[str]) -> list[list[float]]:
    """여러 텍스트의 임베딩을 한 번에 생성한다."""
    response = openai_client.embeddings.create(
        input=texts, model="text-embedding-3-small"
    )
    return [item.embedding for item in response.data]

# 1. 컬렉션 생성
collection = chroma_client.get_or_create_collection(name="kds_structural_ex3")

# 2. 문서 임베딩 및 저장
doc_texts = [doc["text"] for doc in documents]
doc_ids = [doc["id"] for doc in documents]
doc_metadatas = [doc["metadata"] for doc in documents]
doc_embeddings = get_embeddings(doc_texts)

collection.add(
    ids=doc_ids,
    documents=doc_texts,
    metadatas=doc_metadatas,
    embeddings=doc_embeddings
)
print(f"저장 완료: {collection.count()}개 문서\n")

# 3. 검색 수행
for query in queries:
    q_emb = get_embeddings([query])[0]
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )

    print(f"질의: {query}")
    for i in range(len(results["ids"][0])):
        doc_id = results["ids"][0][i]
        dist = results["distances"][0][i]
        meta = results["metadatas"][0][i]
        text = results["documents"][0][i]
        print(f"  [{i+1}] {doc_id} (거리: {dist:.4f}) 조항: {meta['clause']} | 주제: {meta['topic']}")
        print(f"      {text[:70]}...")
    print()

def verify_ex3():
    assert collection.count() == 8, f"문서 수가 8개여야 합니다 (현재: {collection.count()})"
    test_q = get_embeddings(["기둥 철근비"])[0]
    test_r = collection.query(query_embeddings=[test_q], n_results=1, include=["metadatas"])
    assert test_r["metadatas"][0][0]["clause"] == "4.4.1", "기둥 철근비 질의 시 4.4.1이 1위여야 합니다"
    print("Exercise 3 통과!")

verify_ex3()

---
## Exercise 4: RAG 파이프라인 완성

ChromaDB 검색 결과를 Claude에게 전달하여 근거 기반 답변을 생성하는 전체 RAG 파이프라인을 완성하세요.

**요구사항:**
1. `rag_query(question, n_results)` 함수 구현
   - ChromaDB에서 질의와 유사한 문서 검색
   - 검색된 문서를 프롬프트에 포함
   - Claude API로 답변 생성
   - 답변에 KDS 조항 번호가 인용되어야 함
2. 2가지 질의로 테스트

**테스트 질의:**
```python
"500x500 기둥에 8-D25를 배치할 때 철근비는 KDS 기준에 적합한가?"
"보의 전단강도 검토에 필요한 공식을 설명해주세요."
```

**기대 출력:**
```
Q: 500x500 기둥에 8-D25를 배치할 때 철근비는 KDS 기준에 적합한가?

[검색된 조항]
- KDS 4.4.1: 기둥의 종방향 철근비는...
- KDS 4.3.1: 축하중을 받는 부재의...
- KDS 4.2.1: 일반 사항...

A: KDS 4.4.1에 따르면, 기둥의 종방향 철근비는 1% 이상 8% 이하여야 합니다.
   8-D25의 단면적 = 8 × 506.7 = 4,053.6 mm²
   기둥 단면적 = 500 × 500 = 250,000 mm²
   철근비 = 4,053.6 / 250,000 = 0.0162 (1.62%)
   → 1% ≤ 1.62% ≤ 8%이므로 KDS 기준에 적합합니다.
```

In [ ]:
# TODO: rag_query 함수를 구현하고 테스트하세요

test_queries = [
    "500x500 기둥에 8-D25를 배치할 때 철근비는 KDS 기준에 적합한가?",
    "보의 전단강도 검토에 필요한 공식을 설명해주세요.",
]

In [ ]:
# ===== 정답 =====

def rag_query(question: str, n_results: int = 3) -> str:
    """RAG 파이프라인: 검색 → 프롬프트 조립 → LLM 응답 생성"""
    # 1. 검색
    q_emb = get_embeddings([question])[0]
    results = collection.query(
        query_embeddings=[q_emb],
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )

    # 2. 검색 결과 출력
    print("[검색된 조항]")
    context_parts = []
    for i in range(len(results["ids"][0])):
        clause = results["metadatas"][0][i]["clause"]
        text = results["documents"][0][i]
        context_parts.append(f"[KDS {clause}] {text}")
        print(f"  - KDS {clause}: {text[:50]}...")

    context = "\n\n".join(context_parts)

    # 3. 프롬프트 조립
    prompt = f"""다음은 KDS 콘크리트구조 설계기준에서 검색된 관련 조항입니다:

{context}

위 기준을 참고하여 다음 질문에 답변해주세요.
반드시 해당 조항 번호를 인용하세요 (예: KDS 4.4.1에 따르면...).
계산이 필요하면 단계별로 보여주세요.

질문: {question}"""

    # 4. LLM 응답 생성
    message = anthropic_client.messages.create(
        model=model,
        max_tokens=1024,
        temperature=0.1,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

# 테스트 실행
for q in test_queries:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    answer = rag_query(q)
    print(f"\nA: {answer}")

def verify_ex4():
    answer = rag_query("기둥 최소 철근비", n_results=2)
    assert "4.4.1" in answer or "1%" in answer, "답변에 KDS 4.4.1 또는 1%가 포함되어야 합니다"
    print("\nExercise 4 통과!")

verify_ex4()

---
## 건축공학 실습: KDS 조항 기반 지능형 청킹

건축 설계기준 전용 청킹 함수를 구현하세요. 조항 번호(예: 4.2.1)를 인식하여 조항 단위로 분할하고, 각 청크에 메타데이터를 추가합니다.

**요구사항:**
1. `kds_chunk(text)` 함수 구현
   - 조항 번호 패턴 (`숫자.숫자.숫자`) 인식
   - 각 청크에 `clause_id`, `title`, `content`, `source` 메타데이터 포함
2. 제공된 KDS 샘플 텍스트에 적용
3. 결과를 ChromaDB에 저장하고 검색 테스트

**기대 출력:**
```
=== KDS 조항 기반 청킹 ===
[4.2.1] 일반 사항 (87자)
[4.2.2] 강도감소계수 (95자)
[4.3.1] 축하중을 받는 부재 (102자)
[4.4.1] 최소 철근비 (78자)

ChromaDB 저장 완료: 4개 조항
검색 결과 (질의: "강도감소계수"):
  [1] kds-4.2.2 — 강도감소계수...
```

In [ ]:
# KDS 샘플 텍스트
kds_text = """4.2.1 일반 사항
콘크리트구조 부재의 설계는 극한강도설계법에 따른다.
모든 부재는 소요강도 이상의 설계강도를 가져야 한다.

4.2.2 강도감소계수
인장지배 단면의 강도감소계수는 0.85로 한다.
압축지배 단면: 나선철근 부재 0.70, 기타 부재 0.65

4.3.1 축하중을 받는 부재
공칭강도 Pn = 0.80 * [0.85 * fck * (Ag - Ast) + fy * Ast]
여기서 Ag는 전체 단면적, Ast는 철근 단면적이다.

4.4.1 최소 철근비
기둥의 종방향 철근비는 전체 단면적의 1% 이상, 8% 이하로 한다.
최소 4개의 종방향 철근을 배치해야 한다."""

# TODO: kds_chunk 함수를 구현하고, ChromaDB에 저장/검색하세요

In [ ]:
# ===== 정답 =====
import re

def kds_chunk(text: str) -> list[dict]:
    """KDS 기준서 전용 청킹: 조항 번호 기준 분할 + 메타데이터"""
    pattern = r'(\d+\.\d+(?:\.\d+)?)\s+(.+)'
    sections = re.split(r'(?=\d+\.\d+(?:\.\d+)?\s)', text)

    chunks = []
    for section in sections:
        section = section.strip()
        if not section:
            continue
        match = re.match(pattern, section)
        if match:
            chunks.append({
                "clause_id": match.group(1),
                "title": match.group(2).split('\n')[0].strip(),
                "content": section,
                "source": "KDS 14 20 20"
            })
        elif chunks:
            chunks[-1]["content"] += "\n" + section
    return chunks

# 1. 조항 기반 청킹 실행
kds_chunks = kds_chunk(kds_text)
print("=== KDS 조항 기반 청킹 ===")
for c in kds_chunks:
    print(f"[{c['clause_id']}] {c['title']} ({len(c['content'])}자)")

# 2. ChromaDB 저장
kds_collection = chroma_client.get_or_create_collection(name="kds_clause_chunks")

chunk_texts = [c["content"] for c in kds_chunks]
chunk_ids = [f"kds-{c['clause_id']}" for c in kds_chunks]
chunk_metas = [{"clause": c["clause_id"], "title": c["title"], "source": c["source"]} for c in kds_chunks]
chunk_embeddings = get_embeddings(chunk_texts)

kds_collection.add(
    ids=chunk_ids,
    documents=chunk_texts,
    metadatas=chunk_metas,
    embeddings=chunk_embeddings
)
print(f"\nChromaDB 저장 완료: {kds_collection.count()}개 조항")

# 3. 검색 테스트
test_q = "강도감소계수"
q_emb = get_embeddings([test_q])[0]
results = kds_collection.query(
    query_embeddings=[q_emb], n_results=2, include=["documents", "metadatas"]
)
print(f'\n검색 결과 (질의: "{test_q}"):')
for i in range(len(results["ids"][0])):
    meta = results["metadatas"][0][i]
    print(f"  [{i+1}] {results['ids'][0][i]} — {meta['title']}")

def verify_structural():
    assert len(kds_chunks) >= 4, "4개 이상의 조항이 추출되어야 합니다"
    clauses = [c["clause_id"] for c in kds_chunks]
    assert "4.2.2" in clauses, "4.2.2 조항이 포함되어야 합니다"
    assert all("clause_id" in c and "title" in c and "content" in c for c in kds_chunks), "메타데이터 필드 누락"
    print("\n건축공학 실습 통과!")

verify_structural()